In [0]:
catalog = "cinedata_medallion"
land_schema_name = "landing"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

land_schema = f"{catalog}.{land_schema_name}"
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path =  f"/Volumes/{catalog}/{land_schema_name}/inputs"

##About SK generation

I decided to choose the row_number() for the key generation. Despite being slow, it is deterministic, reproducible and readable

##1. Dimension movies

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_movies = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_silver_movies

    # SK key generation, ordered by id_filme
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"{gold_schema}.dim_movies: {df_dim_movies.count()} movies")
display(df_dim_movies.limit(5))

###1.1 Validation

In [0]:
# Check for SK dupes
dupes = df_dim_movies.groupBy("sk_movie_id").count().filter("count > 1").count()
print(f"SKs dupes: {dupes}") 

# Check for id_filme dupes
dupes_nat = df_dim_movies.groupBy("id_filme").count().filter("count > 1").count()
print(f"Natural keys dupes: {dupes_nat}") 

# SK range
df_dim_movies.select(
    F.min("sk_movie_id").alias("min_sk"),
    F.max("sk_movie_id").alias("max_sk"),
    F.count("*").alias("total")
).show()

##2. Dimension genres

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_silver_generos
    .select("nome_genero")
    .distinct()
    # SK key generation by alphabet order
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")))
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"{gold_schema}.dim_genres: {df_dim_genres.count()} genres")
display(df_dim_genres.orderBy("sk_genre_id"))

In [0]:
# SK dupes check
dupes_sk = df_dim_genres.groupBy("sk_genre_id").count().filter("count > 1").count()
print(f"SK dupes: {dupes_sk}")

# nome_genero dupes check
dupes_nome = df_dim_genres.groupBy("nome_genero").count().filter("count > 1").count()
print(f"Genres dupes: {dupes_nome}")

# Range
df_dim_genres.select(
    F.min("sk_genre_id").alias("min_sk"),
    F.max("sk_genre_id").alias("max_sk"),
    F.count("*").alias("total")
).show()

##3. Dimension people

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_silver_pessoas
    
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))

    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    # SK generation, ordered by name + type
    .withColumn(
        "sk_person_id",
        F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa"))
    )
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"{gold_schema}.dim_people: {df_dim_people.count()} people")
display(df_dim_people.limit(10))

In [0]:
# SK dupe check
dupes_sk = df_dim_people.groupBy("sk_person_id").count().filter("count > 1").count()
print(f"SK dupes: {dupes_sk}")

# pair dupes check
dupes_par = (
    df_dim_people
    .groupBy("nome_pessoa", "tipo_pessoa")
    .count()
    .filter("count > 1")
    .count()
)
print(f"Pairs (nome, tipo) dupes: {dupes_par}")

# Distribution by type
display(df_dim_people.groupBy("tipo_pessoa").count().orderBy(F.desc("count")))

# Range
df_dim_people.select(
    F.min("sk_person_id").alias("min_sk"),
    F.max("sk_person_id").alias("max_sk"),
    F.count("*").alias("total")
).show()

##4.Dimension companies

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_companies = (
    df_silver_pessoas
    
    .filter(F.col("tipo_entidade") == "Produtora")
    
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    # SK generation, ordered by name
    .withColumn(
        "sk_company_id",
        F.row_number().over(Window.orderBy("nome_produtora"))
    )
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"{gold_schema}.dim_companies: {df_dim_companies.count()} producers")
display(df_dim_companies.limit(10))

In [0]:
# SK dupe check
dupes_sk = df_dim_companies.groupBy("sk_company_id").count().filter("count > 1").count()
print(f"SK dupes: {dupes_sk}")

# nome dupe check
dupes_nome = df_dim_companies.groupBy("nome_produtora").count().filter("count > 1").count()
print(f"Producers dupes: {dupes_nome}")

# Top 10 producers
display(
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .groupBy("nome_entidade")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

# Range
df_dim_companies.select(
    F.min("sk_company_id").alias("min_sk"),
    F.max("sk_company_id").alias("max_sk"),
    F.count("*").alias("total")
).show()

##5. Dimension reviews

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

# Agrega reviews por filme
df_agg = (
    df_silver_reviews
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
)

# Join com dim_movies para trazer sk_movie_id
df_dim_reviews = (
    df_agg
    .join(
        df_dim_movies.select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner"
    )
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"{gold_schema}.dim_reviews: {df_dim_reviews.count()} movies with reviews")
display(df_dim_reviews.limit(10))

In [0]:
# SK dupe check
dupes_sk = df_dim_reviews.groupBy("sk_review_id").count().filter("count > 1").count()
print(f"SK dupes: {dupes_sk}")

# sk_movie_id dupe check
dupes_movie = df_dim_reviews.groupBy("sk_movie_id").count().filter("count > 1").count()
print(f"sk_movie_id dupes: {dupes_movie}")

# Orphan FKs
orfas = df_dim_reviews.join(
    df_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti"
).count()
print(f"orphans sk_movie_id: {orfas}")

# Statistics
df_dim_reviews.select(
    F.count("*").alias("total_filmes"),
    F.sum("qtd_avaliacoes_usuarios").alias("total_reviews"),
    F.round(F.avg("nota_media_usuarios"), 2).alias("media_geral"),
    F.min("nota_media_usuarios").alias("nota_min"),
    F.max("nota_media_usuarios").alias("nota_max"),
).show()

##6. Bridges

###6.1 Movie genre bridge

In [0]:
from pyspark.sql import functions as F

df_silver_generos = spark.table(f"{silver_schema}.tb_generos")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

df_bridge_movie_genre = (
    df_silver_generos
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(df_dim_genres.select("sk_genre_id", "nome_genero"), on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

print(f"{gold_schema}.bridge_movie_genre: {df_bridge_movie_genre.count()} pairs")

In [0]:
dupes = df_bridge_movie_genre.groupBy("sk_movie_id", "sk_genre_id").count().filter("count > 1").count()
orfas_movie = df_bridge_movie_genre.join(df_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti").count()
orfas_genre = df_bridge_movie_genre.join(df_dim_genres.select("sk_genre_id"), on="sk_genre_id", how="left_anti").count()

print(f"Pairs dupes: {dupes}")
print(f"orphans sk_movie_id: {orfas_movie}")
print(f"orphans sk_genre_id: {orfas_genre}")

df_bridge_movie_genre.select(
    F.count("*").alias("total_pares"),
    F.countDistinct("sk_movie_id").alias("filmes_distintos"),
    F.countDistinct("sk_genre_id").alias("generos_distintos"),
).show()

###6.2 Movie person bridge

In [0]:
df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

df_bridge_movie_person = (
    df_silver_pessoas
    
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    # movie SK
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    # person SK 
    .join(
        df_dim_people.select("sk_person_id", "nome_pessoa", "tipo_pessoa"),
        on=["nome_pessoa", "tipo_pessoa"],
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

print(f"{gold_schema}.bridge_movie_person: {df_bridge_movie_person.count()} pairs")

In [0]:
dupes = df_bridge_movie_person.groupBy("sk_movie_id", "sk_person_id").count().filter("count > 1").count()
orfas_movie = df_bridge_movie_person.join(df_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti").count()
orfas_person = df_bridge_movie_person.join(df_dim_people.select("sk_person_id"), on="sk_person_id", how="left_anti").count()

print(f"Pairs dupes: {dupes}")
print(f"Orphans sk_movie_id: {orfas_movie}")
print(f"Orphans sk_person_id: {orfas_person}")

df_bridge_movie_person.select(
    F.count("*").alias("total_pares"),
    F.countDistinct("sk_movie_id").alias("filmes_distintos"),
    F.countDistinct("sk_person_id").alias("pessoas_distintas"),
).show()

###6.3 Bridge movie company

In [0]:
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

df_bridge_movie_company = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        df_dim_companies.select("sk_company_id", "nome_produtora"),
        on="nome_produtora",
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"{gold_schema}.bridge_movie_company: {df_bridge_movie_company.count()} pairs")

In [0]:
dupes = df_bridge_movie_company.groupBy("sk_movie_id", "sk_company_id").count().filter("count > 1").count()
orfas_movie = df_bridge_movie_company.join(df_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti").count()
orfas_company = df_bridge_movie_company.join(df_dim_companies.select("sk_company_id"), on="sk_company_id", how="left_anti").count()

print(f"Pairs dupes: {dupes}")
print(f"Orphan sk_movie_id: {orfas_movie}")
print(f"Orphan sk_company_id: {orfas_company}")

df_bridge_movie_company.select(
    F.count("*").alias("total_pares"),
    F.countDistinct("sk_movie_id").alias("filmes_distintos"),
    F.countDistinct("sk_company_id").alias("produtoras_distintas"),
).show()

##7. Fact movies performance

In [0]:
from pyspark.sql import functions as F

df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_silver_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_eng = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact_movies_performance = (
    df_dim_movies
    # Finance metrics
    .join(
        df_silver_fin.select(
            "id_filme",
            "orcamento_usd", "receita_usd", "lucro_usd",
            "orcamento_brl", "receita_brl", "lucro_brl"
        ),
        on="id_filme",
        how="left"
    )
    # Engajement metrics
    .join(
        df_silver_eng.select(
            "id_filme",
            "popularidade",
            "nota_media_tmdb", "qtd_votos_tmdb",
            "nota_media_imbd", "qtd_votos_imbd"
        ),
        on="id_filme",
        how="left"
    )
    # BRL ---> BLR
    .withColumnRenamed("orcamento_brl", "orcamento_blr")
    .withColumnRenamed("receita_brl", "receita_blr")
    .withColumnRenamed("lucro_brl", "lucro_blr")
    # Select columns
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_blr", "receita_blr", "lucro_blr",
        "popularidade",
        "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imbd", "qtd_votos_imbd",
    )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"{gold_schema}.fact_movies_performance: {df_fact_movies_performance.count()} movies")
display(df_fact_movies_performance.limit(10))

In [0]:
# Check for SK dupes
dupes = df_fact_movies_performance.groupBy("sk_movie_id").count().filter("count > 1").count()
print(f"sk_movie_id duplicados: {dupes}")  # Esperado: 0

# Check for orphan FKs
orfas = df_fact_movies_performance.join(
    df_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti"
).count()
print(f"sk_movie_id órfãos: {orfas}")

# Line amount = dim_movies total?
print(f"Total fact: {df_fact_movies_performance.count()}")
print(f"Total dim_movies: {df_dim_movies.count()}")

# Metrics
df_fact_movies_performance.select(
    F.count("*").alias("total"),
    F.count("orcamento_usd").alias("com_orcamento"),
    F.count("receita_usd").alias("com_receita"),
    F.count("popularidade").alias("com_popularidade"),
    F.count("nota_media_tmdb").alias("com_nota_tmdb"),
    F.count("nota_media_imbd").alias("com_nota_imbd"),
).show()

##8. Gen AI movie context

###8.1 Agreggate actors and directors by movie

In [0]:
from pyspark.sql import functions as F

df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

df_atores_por_filme = (
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores")
    )
)

df_diretor_por_filme = (
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        F.first("nome_pessoa").alias("diretor")
    )
)

display(df_atores_por_filme.limit(5))
display(df_diretor_por_filme.limit(5))

###8.2 Build context table

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")

df_genai_context = (
    df_dim_movies

    .join(
        df_fact.select("sk_movie_id", "orcamento_usd", "receita_usd", "orcamento_blr", "receita_blr"),
        on="sk_movie_id",
        how="left"
    )
    # Actors
    .join(df_atores_por_filme, on="sk_movie_id", how="left")
    # Directors
    .join(df_diretor_por_filme, on="sk_movie_id", how="left")
    
    # Safeguard to avoid strings becoming NULL
    .withColumn("titulo_ctx",   F.coalesce(F.col("titulo"), F.lit("Título Desconhecido")))
    .withColumn("ano_ctx",      F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano desconhecido")))
    .withColumn("receita_ctx",  F.coalesce(F.col("receita_usd").cast("string"), F.lit("valor não informado")))
    .withColumn("orcamento_ctx",F.coalesce(F.col("orcamento_usd").cast("string"), F.lit("valor não informado")))
    .withColumn("atores_ctx",   F.coalesce(F.col("atores"), F.lit("elenco não informado")))
    .withColumn("diretor_ctx",  F.coalesce(F.col("diretor"), F.lit("diretor não informado")))
    .withColumn("sinopse_ctx",  F.coalesce(F.col("sinopse"), F.lit("Sinopse não disponível.")))
    
    # Final concatenation
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("titulo_ctx"),
            F.lit(", lançado no ano de "), F.col("ano_ctx"),
            F.lit(", faturou "), F.col("receita_ctx"),
            F.lit(" e teve um custo de "), F.col("orcamento_ctx"),
            F.lit(". Estrelado por "), F.col("atores_ctx"),
            F.lit(" e dirigido por "), F.col("diretor_ctx"),
            F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_ctx"),
            F.lit(".")
        )
    )
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        "llm_context_document"
    )
)

df_genai_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")

print(f"{gold_schema}.gold_genai_movies_context: {df_genai_context.count()} movies")

###8.3 Validation

In [0]:
# Line amount
print(f"Total: {df_genai_context.count()}")

# Check for null and empty strings
nulos = df_genai_context.filter(F.col("llm_context_document").isNull()).count()
print(f"Documentos nulos: {nulos}") 

vazios = df_genai_context.filter(F.length(F.col("llm_context_document")) < 30).count()
print(f"Documentos muito curtos: {vazios}")

display(df_genai_context.limit(3))

##9. Desafio Analytics

###9.1 Limit date setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

data_limite = (
    df_dim_movies
    .filter(F.col("data_lancamento").isNotNull())
    .filter(F.col("data_lancamento") <= F.current_date())     # ignore future
    .filter(F.col("status_filme") == "Lançado")                # ignores non-releases
    .agg(F.max("data_lancamento").alias("max_data"))
    .collect()[0]["max_data"]
)

print(f"Limit date (most recent release): {data_limite}")
print(f"2 year cutoff: {data_limite.year - 2}")
print(f"5 year cutoff: {data_limite.year - 5}")

###9.2 Total revenue

In [0]:
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")

display(
    df_fact.agg(
        F.round(F.sum("receita_blr"), 2).alias("receita_total_brl"),
        F.count("receita_blr").alias("filmes_com_receita"),
    )
)

###9.3 Top 5 movies by popularity

In [0]:
display(
    df_dim_movies
    .join(df_fact, on="sk_movie_id", how="inner")
    .select("titulo", "popularidade")
    .filter(F.col("popularidade").isNotNull())
    .orderBy(F.desc("popularidade"))
    .limit(5)
)

###9.4 Movies by genre

In [0]:
df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

display(
    df_bridge_genre
    .join(df_dim_genres, on="sk_genre_id", how="inner")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"))
)

###9.5 Top 10 movies by revenue

In [0]:
display(
    df_dim_movies
    .join(df_fact, on="sk_movie_id", how="inner")
    .filter(F.col("receita_usd").isNotNull())
    .select(
        "titulo",
        "receita_usd",
        "receita_blr",
        F.rank().over(Window.orderBy(F.desc("receita_usd"))).alias("ranking")
    )
    .orderBy("ranking")
    .limit(10)
)

###9.6 Actor with most appereances in the last 2 yeas

In [0]:
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

ano_limite_2 = data_limite.year - 2

display(
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .join(df_dim_movies.select("sk_movie_id", "ano_lancamento"), on="sk_movie_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .filter(F.col("ano_lancamento") >= ano_limite_2)
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"))
    .limit(10)
)

###9.7 Producers with the highest profit in the last 5 years

In [0]:
df_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

ano_limite_5 = data_limite.year - 5

display(
    df_bridge_company
    .join(df_dim_companies, on="sk_company_id", how="inner")
    .join(df_dim_movies.select("sk_movie_id", "ano_lancamento"), on="sk_movie_id", how="inner")
    .join(df_fact.select("sk_movie_id", "lucro_usd", "lucro_blr"), on="sk_movie_id", how="inner")
    .filter(F.col("ano_lancamento") >= ano_limite_5)
    .filter(F.col("lucro_usd").isNotNull())
    .groupBy("nome_produtora")
    .agg(
        F.round(F.sum("lucro_usd"), 2).alias("lucro_total_usd"),
        F.round(F.sum("lucro_blr"), 2).alias("lucro_total_blr"),
        F.countDistinct("sk_movie_id").alias("qtd_filmes")
    )
    .orderBy(F.desc("lucro_total_usd"))
    .limit(10)
)